In [1]:
import polars as pl

In [2]:
folder = 'D:\\Pythonfiles\\northwind\\'

names = [
    "categories", "customer_customer_demo", "customer_demographics",
    "customers", "employee_territories", "employees", "order_details",
    "orders", "products", "region", "shippers", "suppliers",
    "territories", "us_states"
]

for name in names:
    globals()[name] = pl.read_csv(f"{folder}{name}.csv", truncate_ragged_lines=True, separator=';')

In [108]:
suppliers

supplier_id,company_name,contact_name,contact_title,address,city,region,postal_code,country,phone,fax,homepage
i64,str,str,str,str,str,str,str,str,str,str,str
1,"""Exotic Liquids""","""Charlotte Cooper""","""Purchasing Manager""","""49 Gilbert St.""","""London""",null,"""EC1 4SD""","""UK""","""(171) 555-2222""",null,null
2,"""New Orleans Cajun Delights""","""Shelley Burke""","""Order Administrator""","""P.O. Box 78934""","""New Orleans""","""LA""","""70117""","""USA""","""(100) 555-4822""",null,"""#CAJUN.HTM#"""
3,"""Grandma Kelly's Homestead""","""Regina Murphy""","""Sales Representative""","""707 Oxford Rd.""","""Ann Arbor""","""MI""","""48104""","""USA""","""(313) 555-5735""","""(313) 555-3349""",null
4,"""Tokyo Traders""","""Yoshi Nagase""","""Marketing Manager""","""9-8 Sekimai Musashino-shi""","""Tokyo""",null,"""100""","""Japan""","""(03) 3555-5011""",null,null
5,"""Cooperativa de Quesos 'Las Cab…","""Antonio del Valle Saavedra""","""Export Administrator""","""Calle del Rosal 4""","""Oviedo""","""Asturias""","""33007""","""Spain""","""(98) 598 76 54""",null,null
…,…,…,…,…,…,…,…,…,…,…,…
25,"""Ma Maison""","""Jean-Guy Lauzon""","""Marketing Manager""","""2960 Rue St. Laurent""","""Montréal""","""Québec""","""H1J 1C3""","""Canada""","""(514) 555-9022""",null,null
26,"""Pasta Buttini s.r.l.""","""Giovanni Giudici""","""Order Administrator""","""Via dei Gelsomini, 153""","""Salerno""",null,"""84100""","""Italy""","""(089) 6547665""","""(089) 6547667""",null
27,"""Escargots Nouveaux""","""Marie Delamare""","""Sales Manager""","""22, rue H. Voiron""","""Montceau""",null,"""71300""","""France""","""85.57.00.07""",null,null


Dentre os top 10 que mais compram, 2 deles não possuem fornecedor próprio.

In [62]:
#Entendendo se há clientes de países que não possuem fornecedores

customer_countries = customers.select("country").unique()
supplier_countries = suppliers.select("country").unique()

expansion_candidates = customer_countries.join(supplier_countries, on="country", how="anti")
expansion_candidates

country
str
"""Mexico"""
"""Ireland"""
"""Venezuela"""
"""Argentina"""
"""Belgium"""
"""Poland"""
"""Austria"""
"""Portugal"""
"""Switzerland"""


In [119]:
supplier_countries = suppliers.select("country").unique().to_series()

orders.group_by('ship_country'
    ).agg(
        pl.len().alias('count')
        ).sort(by = 'count', descending = True
            ).head(10).filter(
                ~pl.col('ship_country').is_in(supplier_countries)
                ).rename({
                    'ship_country':'País - Encomenda',
                    'count':'Contagem'
                })

<positron-console-cell-119>:7: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.


País - Encomenda,Contagem
str,u32
"""Venezuela""",46
"""Austria""",40
"""Mexico""",28


Produtos que deveriam ser descontinuados

In [138]:
order_details.group_by("product_id"
    ).agg(
        pl.sum("quantity").alias("Quantidade")
        ).with_columns(
           (pl.col("Quantidade") / pl.sum("Quantidade") * 100).alias("Representação")
            ).join(products.select(
                    ["product_id", "product_name"]
                        ), on="product_id"
                            ).filter(pl.col('Representação') < 1
                                ).select(["product_name","Quantidade", "Representação"]
                                    ).sort('Representação').head(10).rename({
                                        'product_name':'Produto'
                                        })

Produto,Quantidade,Representação
str,i64,f64
"""Mishi Kobe Niku""",95,0.185124
"""Genen Shouyu""",122,0.237738
"""Gravad lax""",125,0.243584
"""Chocolade""",138,0.268917
"""Laughing Lumberjack Lager""",184,0.358556
"""Valkoinen suklaa""",235,0.457938
"""Louisiana Hot Spiced Okra""",239,0.465733
"""Röd Kaviar""",293,0.570961
"""Mascarpone Fabioli""",297,0.578756


Clientes que faziam compras com frequência e estou ausentes há muito tempo

In [161]:
ordem_mais_recente = (
    orders.group_by("customer_id")
    .agg(pl.col("order_date").max().alias("data_ult_ordem"))
)

media_sem_compras = (
    orders.sort(["customer_id", "order_date"])
    .with_columns(
        pl.col("order_date").shift(1).over("customer_id").alias("ordem_ant")
    )
    .with_columns(
        (pl.col("order_date").str.to_date() - pl.col("ordem_ant").str.to_date()).alias("sem_compras")
    )
    .group_by("customer_id")
    .agg(pl.col("sem_compras").mean().dt.total_days().alias("med_sem_compras"))
)

data_max = orders["order_date"].max() # Para uma análise do dia a dia, isso seria a data de hoje

order_counts = (
    orders.group_by("customer_id")
    .agg(pl.col('order_id').n_unique().alias("order_count"))
)


ordem_mais_recente.join(
        media_sem_compras, on="customer_id"
        ).join(
            customers.select(["customer_id", "company_name", "country"]), on="customer_id"
            ).join(
                order_counts, on="customer_id"
                ).filter(
                    pl.col("order_count") >= 5,
                    (pl.lit(data_max).str.to_date() - pl.col("data_ult_ordem").str.to_date()).dt.total_days() >= 90,
                    pl.col('med_sem_compras') < 90
                    ).rename({
                        'customer_id':'Id Cliente',
                        'company_name':'Nome Cliente',
                        'country':'País',
                        'order_count':'Contagem'
                        })


Id Cliente,data_ult_ordem,med_sem_compras,Nome Cliente,País,Contagem
str,str,i64,str,str,u32
"""SEVES""","""1998-02-04""",55,"""Seven Seas Imports""","""UK""",9
"""VICTE""","""1998-01-23""",62,"""Victuailles en stock""","""France""",10
"""FOLIG""","""1997-12-22""",87,"""Folies gourmandes""","""France""",5
"""BLONP""","""1998-01-12""",53,"""Blondesddsl père et fils""","""France""",11
"""FAMIA""","""1997-10-31""",59,"""Familia Arquibaldo""","""Brazil""",7
"""HUNGC""","""1997-09-08""",69,"""Hungry Coyote Import Store""","""USA""",5
"""ANTON""","""1998-01-28""",71,"""Antonio Moreno Taquería""","""Mexico""",7
"""MEREP""","""1997-10-30""",31,"""Mère Paillarde""","""Canada""",13


Piores Fornecedores

In [16]:
orders.select('ship_via').unique()

ship_via
i64
1
3
2


In [20]:
orders.filter(pl.col("shipped_date").is_not_null()
    ).with_columns([
        pl.col("required_date").str.to_date(),
        pl.col("shipped_date").str.to_date()
        ]).with_columns(
            (pl.col("shipped_date") - pl.col("required_date")).dt.total_days().alias("dias_de_atraso")
            ).filter(pl.col("dias_de_atraso") > 0
                ).group_by("ship_via").agg([
                    pl.len().alias("Ordens atrasadas"),
                    pl.col("dias_de_atraso").mean().alias("media_de_atraso"),
                    pl.col("dias_de_atraso").max().alias("maior_atraso")
                    ]).join(shippers.select(["shipper_id", "company_name"]), left_on="ship_via", right_on="shipper_id"
                        ).sort("media_de_atraso", descending=True
                            ).rename({
                                'company_name' : 'Nome Fornecedor'
                                }).select(['Nome Fornecedor', "Ordens atrasadas", "media_de_atraso", "maior_atraso"]
                                    )


Nome Fornecedor,Ordens atrasadas,media_de_atraso,maior_atraso
str,u32,f64,i64
"""Speedy Express""",12,8.083333,18
"""United Package""",16,5.5625,23
"""Federal Shipping""",9,5.555556,18
